In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model

# Sample data - English to Russian translations
english_sentences = ['hello', 'how are you?', 'i am fine', 'what is your name?', 'my name is John']
russian_sentences = ['привет', 'как ты?', 'я в порядке', 'как тебя зовут?', 'меня зовут Джон']

# Tokenization
english_tokenizer = keras.preprocessing.text.Tokenizer(filters='')
english_tokenizer.fit_on_texts(english_sentences)
english_vocab_size = len(english_tokenizer.word_index) + 1
english_seq = english_tokenizer.texts_to_sequences(english_sentences)

russian_tokenizer = keras.preprocessing.text.Tokenizer(filters='')
russian_tokenizer.fit_on_texts(russian_sentences)
russian_vocab_size = len(russian_tokenizer.word_index) + 1
russian_seq = russian_tokenizer.texts_to_sequences(russian_sentences)

# Add special tokens '<start>' and '<end>' to the Russian tokenizer
russian_tokenizer.word_index['<start>'] = russian_vocab_size
russian_tokenizer.word_index['<end>'] = russian_vocab_size + 1
russian_vocab_size += 2

# Padding sequences
max_length = max(len(seq) for seq in english_seq)
english_seq = keras.preprocessing.sequence.pad_sequences(english_seq, maxlen=max_length, padding='post')
russian_seq = keras.preprocessing.sequence.pad_sequences(russian_seq, maxlen=max_length, padding='post')

# Define model
latent_dim = 256

# Encoder
encoder_inputs = Input(shape=(None,))
encoder_embedding = keras.layers.Embedding(english_vocab_size, latent_dim, mask_zero=True)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
decoder_embedding = keras.layers.Embedding(russian_vocab_size, latent_dim, mask_zero=True)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
decoder_dense = Dense(russian_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Define the model that will turn
# `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# Compile model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train model
model.fit([english_seq, russian_seq[:, :-1]], russian_seq[:, 1:], batch_size=64, epochs=50, validation_split=0.2)

# Inference model
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs, state_h, state_c = decoder_lstm(decoder_embedding, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states)

# Reverse-lookup token index to decode sequences back to something readable
reverse_input_index = {v: k for k, v in english_tokenizer.word_index.items()}
reverse_target_index = {v: k for k, v in russian_tokenizer.word_index.items()}

def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = russian_tokenizer.word_index['<start>']

    stop_condition = False
    decoded_sentence = ''
    sampled_word = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        if sampled_token_index in reverse_target_index:
            sampled_word = reverse_target_index[sampled_token_index]
            if sampled_word != '<end>':
                decoded_sentence += sampled_word + ' '
        else:
            decoded_sentence += '<unknown> '

        # Exit condition: either hit max length or find stop token
        if sampled_word == '<end>' or len(decoded_sentence.split()) > max_length:
            stop_condition = True

        # Update the target sequence (of length 1)
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # Update states
        states_value = [h, c]

    return decoded_sentence

# Test the model
for seq_index in range(len(english_seq)):
    input_seq = english_seq[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print('-')
    print('Input sentence:', english_sentences[seq_index])
    print('Decoded sentence:', decoded_sentence)

Epoch 1/50
1/1 [==============================] - 14s 14s/step - loss: 2.6415 - accuracy: 0.0000e+00 - val_loss: 2.6306 - val_accuracy: 0.0000e+00
Epoch 2/50
1/1 [==============================] - 0s 78ms/step - loss: 2.6051 - accuracy: 0.5556 - val_loss: 2.6277 - val_accuracy: 0.3333
Epoch 3/50
1/1 [==============================] - 0s 87ms/step - loss: 2.5677 - accuracy: 0.6667 - val_loss: 2.6242 - val_accuracy: 0.3333
Epoch 4/50
1/1 [==============================] - 0s 73ms/step - loss: 2.5273 - accuracy: 0.6667 - val_loss: 2.6196 - val_accuracy: 0.3333
Epoch 5/50
1/1 [==============================] - 0s 79ms/step - loss: 2.4821 - accuracy: 0.6667 - val_loss: 2.6136 - val_accuracy: 0.3333
Epoch 6/50
1/1 [==============================] - 0s 74ms/step - loss: 2.4300 - accuracy: 0.6667 - val_loss: 2.6057 - val_accuracy: 0.3333
Epoch 7/50
1/1 [==============================] - 0s 80ms/step - loss: 2.3686 - accuracy: 0.5556 - val_loss: 2.5953 - val_accuracy: 0.3333
Epoch 8/50
1/1 [===